In [6]:
import random
import time
import os

DIRECOES = ['N', 'E', 'S', 'W']          # ordem horária
DELTA = {'N': (0, -1), 'E': (1, 0), 'S': (0, 1), 'W': (-1, 0)}
OPOSTA = {'N': 'S', 'S': 'N', 'E': 'W', 'W': 'E'}


class Robo:
    def __init__(self, labirinto, largura, altura,
                 inicio=(0, 0), objetivo=None, direcao='E'):
        self.labirinto = labirinto
        self.largura = largura
        self.altura = altura
        self.x, self.y = inicio
        self.direcao = direcao
        self.objetivo = objetivo if objetivo else (largura - 1, altura - 1)
        self.passos = 0

    #  sensores
    def tem_parede_frente(self):
        return self.labirinto[(self.x, self.y)][self.direcao]

    def tem_parede_direita(self):
        i = DIRECOES.index(self.direcao)
        direita = DIRECOES[(i + 1) % 4]
        return self.labirinto[(self.x, self.y)][direita]

    def tem_parede_esquerda(self):
        i = DIRECOES.index(self.direcao)
        esquerda = DIRECOES[(i - 1) % 4]
        return self.labirinto[(self.x, self.y)][esquerda]

    def tem_parede_atras(self):
        i = DIRECOES.index(self.direcao)
        atras = DIRECOES[(i + 2) % 4]
        return self.labirinto[(self.x, self.y)][atras]
 
 
    def paredes_ao_redor(self):
        return dict(self.labirinto[(self.x, self.y)])

    def chegou_no_objetivo(self):
        return (self.x, self.y) == self.objetivo

    # ações
    def anda_reto(self):
        # anda se não tiver parede
        if not self.tem_parede_frente():
            dx, dy = DELTA[self.direcao]
            self.x += dx
            self.y += dy
        self.passos += 1

    def vira_direita(self):
        i = DIRECOES.index(self.direcao)
        self.direcao = DIRECOES[(i + 1) % 4]

    def vira_esquerda(self):
        i = DIRECOES.index(self.direcao)
        self.direcao = DIRECOES[(i - 1) % 4]

    def vira_180(self):
        self.vira_direita()
        self.vira_direita()

    def anda_em_circulo(self, n=8):
        for i in range(n):
            self.anda_reto()
            self.vira_direita()
            
    def recua(self):
        i = DIRECOES.index(self.direcao)
        direcao_oposta = DIRECOES[(i + 2) % 4]
        if not self.labirinto[(self.x, self.y)][direcao_oposta]:
            dx, dy = DELTA[direcao_oposta]
            self.x += dx
            self.y += dy
        self.passos += 1

    def anda_aleatorio(self, n=1):
        for i in range(n):
            livres = [d for d in DIRECOES if not self.labirinto[(self.x, self.y)][d]]
            if livres:
                self.direcao = random.choice(livres)
                self.anda_reto()

    def reset(self, inicio=(0, 0), direcao='E'):
        self.x, self.y = inicio
        self.direcao = direcao
        self.passos = 0
    
# algoritmo que combina os sensores com as ações
def segue_parede(robo, max_passos=3000, visualizar=False, atraso=0.03):
    while not robo.chegou_no_objetivo() and robo.passos < max_passos:
        if not robo.tem_parede_direita():
            robo.vira_direita()
            robo.anda_reto()
        elif not robo.tem_parede_frente():
            robo.anda_reto()
        elif not robo.tem_parede_esquerda():
            robo.vira_esquerda()
            robo.anda_reto()
        else:
            robo.vira_180()
            robo.anda_reto()

        if visualizar:
            desenha_labirinto(robo)
            time.sleep(atraso)

    return robo.chegou_no_objetivo()

